<a href="https://colab.research.google.com/github/JoThePOkeMOn/multimodal-speech-emotion-recognition/blob/main/data_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from google.colab import drive

print("--- Initializing Colab Environment ---")

# 1. Mount Google Drive (Forces a refresh to ensure it sees your newest files)
drive.mount('/content/drive', force_remount=True)

# 2. Define the permanent project directories with the NEW name
PROJECT_DIR = '/content/drive/MyDrive/Emotion_Recognition_Project'
DATASET_DIR = os.path.join(PROJECT_DIR, 'TESS_Dataset')
FEATURES_DIR = os.path.join(PROJECT_DIR, 'Features')

# 3. Create the folders if they don't exist yet
os.makedirs(DATASET_DIR, exist_ok=True)
os.makedirs(FEATURES_DIR, exist_ok=True)

# 4. Safely navigate to the main project folder
if os.path.exists(PROJECT_DIR):
    os.chdir(PROJECT_DIR)
    print(f"✅ SUCCESS: Project directories set up and terminal locked to: {os.getcwd()}")

    # List the files so you can visually confirm your folders are there
    print("\n--- Project Contents ---")
    os.system('ls -la')
else:
    print(f"❌ ERROR: Could not find or create {PROJECT_DIR}.")

--- Initializing Colab Environment ---
Mounted at /content/drive
✅ SUCCESS: Project directories set up and terminal locked to: /content/drive/MyDrive/Emotion_Recognition_Project

--- Project Contents ---


In [ ]:
import os
from google.colab import userdata

print("--- Initializing Kaggle Connection ---")

# 1. Fetch credentials securely from Colab's Secrets Vault
try:
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    print("✅ Kaggle credentials loaded securely.")
except userdata.SecretNotFoundError:
    print("❌ ERROR: Kaggle secrets not found!")
    print("Please click the '🔑 Secrets' icon on the left sidebar and add KAGGLE_USERNAME and KAGGLE_KEY.")
    raise

# 2. Download the dataset directly into your Google Drive project folder
print(f"Downloading TESS dataset into {PROJECT_DIR}...")
!kaggle datasets download -d ejlok1/toronto-emotional-speech-set-tess -p {PROJECT_DIR}

# 3. Unzip the dataset into your specific Dataset folder
print("Unzipping directly to Google Drive (this may take a minute)...")
!unzip -q {PROJECT_DIR}/toronto-emotional-speech-set-tess.zip -d {DATASET_DIR}

# 4. Clean up the zip file to save Drive space
print("Cleaning up temporary zip files...")
!rm {PROJECT_DIR}/toronto-emotional-speech-set-tess.zip

print("✅ Download and extraction complete! Your audio files are ready.")

--- Initializing Kaggle Connection ---
✅ Kaggle credentials loaded securely.
Dataset URL: https://www.kaggle.com/datasets/ejlok1/toronto-emotional-speech-set-tess
License(s): Attribution-NonCommercial-NoDerivatives 4.0 International (CC BY-NC-ND 4.0)
100% 428M/428M [00:02<00:00, 160MB/s]

Unzipping directly to Google Drive (this may take a minute)...
Cleaning up temporary zip files...
✅ Download and extraction complete! Your audio files are ready.


In [ ]:
import os
import pandas as pd
from sklearn.preprocessing import LabelEncoder

print("--- Generating Master Metadata CSV ---")

data = []

# Walk through the Google Drive dataset folder
for root, dirs, files in os.walk(DATASET_DIR):
    for file in files:
        if file.endswith(".wav"):
            file_path = os.path.join(root, file)

            # TESS filenames are formatted as: [Actor]_[Word]_[Emotion].wav
            # Example: YAF_dog_angry.wav
            parts = file.split('_')

            # 1. Extract emotion from filename
            emotion = parts[-1].split('.')[0].lower()
            if emotion == 'ps':
                emotion = 'pleasant_surprise'

            # 2. Extract the target word and build the transcript
            # We wrap it in the actual spoken phrase to give RoBERTa semantic context
            if len(parts) >= 3:
                target_word = parts[1].lower()
                transcript = f"Say the word {target_word}."
            else:
                transcript = ""

            data.append({
                "file_path": file_path,
                "transcript": transcript, # Critical for the Text Pipeline
                "emotion": emotion
            })

df = pd.DataFrame(data)

# Convert text labels to integers (0 through 6)
encoder = LabelEncoder()
df['label_id'] = encoder.fit_transform(df['emotion'])

# Save the master CSV permanently to your Drive
csv_path = os.path.join(PROJECT_DIR, 'tess_metadata.csv')
df.to_csv(csv_path, index=False)

print(f"✅ Successfully saved metadata for {len(df)} files to {csv_path}")

# Print the label mapping so you know which number equals which emotion
print("\n--- Emotion ID Mapping ---")
for i, class_name in enumerate(encoder.classes_):
    print(f"ID {i} : {class_name}")

print("\n--- Dataset Preview ---")
display(df.head())

--- Generating Master Metadata CSV ---
✅ Successfully saved metadata for 5600 files to /content/drive/MyDrive/Emotion_Recognition_Project/tess_metadata.csv

--- Emotion ID Mapping ---
ID 0 : angry
ID 1 : disgust
ID 2 : fear
ID 3 : happy
ID 4 : neutral
ID 5 : pleasant_surprise
ID 6 : sad

--- Dataset Preview ---


,file_path,transcript,emotion,label_id
0,/content/drive/MyDrive/Emotion_Recognition_Pro...,Say the word back.,fear,2
1,/content/drive/MyDrive/Emotion_Recognition_Pro...,Say the word bar.,fear,2
2,/content/drive/MyDrive/Emotion_Recognition_Pro...,Say the word base.,fear,2
3,/content/drive/MyDrive/Emotion_Recognition_Pro...,Say the word bath.,fear,2
4,/content/drive/MyDrive/Emotion_Recognition_Pro...,Say the word bean.,fear,2


In [ ]:
# CELL 4
# Install necessary audio and NLP libraries
!pip install -q transformers librosa

import librosa
import numpy as np
import torch
from transformers import AutoTokenizer

print("--- Initializing Extractors and Hyperparameters ---")

# --- HYPERPARAMETERS ---
# Audio Settings
SAMPLE_RATE = 16000
N_MFCC = 40
# Force all audio matrices to have exactly 130 time steps (approx 4 seconds)
TARGET_TIME_STEPS = 130

# Text Settings
# The phrase is "Say the word [target]", so 16 tokens is plenty
MAX_TEXT_LENGTH = 16

# ---------------------------------------------------------
# INITIALIZE TOKENIZER (Must exactly match the TextEncoder in model.py)
# ---------------------------------------------------------
model_name = "distilroberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

print(f"✅ Tokenizer '{model_name}' and hyperparameters loaded successfully.")

--- Initializing Extractors and Hyperparameters ---


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Tokenizer 'distilroberta-base' and hyperparameters loaded successfully.


In [ ]:
# CELL 5
import os
import pandas as pd
import torch
import librosa
import numpy as np
from tqdm.notebook import tqdm
import warnings

# Suppress annoying librosa PySoundFile warnings to keep the terminal clean
warnings.filterwarnings("ignore")

print("--- Starting Multimodal Feature Extraction ---")
print("Note: Saving thousands of files to Google Drive can take 15-30 minutes.")
print("Please leave this tab open until it finishes!")

# 1. Reload the master CSV to ensure we have the newest version
csv_path = os.path.join(PROJECT_DIR, 'tess_metadata.csv')
df = pd.read_csv(csv_path)

# 2. Start the extraction loop
for index, row in tqdm(df.iterrows(), total=len(df), desc="Processing Files"):
    try:
        wav_path = row['file_path']

        # Pull the exact text we generated in Cell 3!
        transcript = str(row['transcript'])
        base_name = os.path.basename(wav_path).replace('.wav', '')

        # -----------------------------------------
        # A. AUDIO PROCESSING (Librosa)
        # -----------------------------------------
        y, sr = librosa.load(wav_path, sr=SAMPLE_RATE)

        # Trim dead silence from ends
        y_trimmed, _ = librosa.effects.trim(y, top_db=20)

        # Extract MFCCs
        mfccs = librosa.feature.mfcc(y=y_trimmed, sr=sr, n_mfcc=N_MFCC)
        mfccs = mfccs.T  # Transpose to (time_steps x features)

        # Pad or truncate to force shape: (130, 40)
        current_steps = mfccs.shape[0]
        if current_steps < TARGET_TIME_STEPS:
            pad_width = TARGET_TIME_STEPS - current_steps
            mfccs = np.pad(mfccs, ((0, pad_width), (0, 0)), mode='constant')
        else:
            mfccs = mfccs[:TARGET_TIME_STEPS, :]

        audio_tensor = torch.tensor(mfccs, dtype=torch.float32)

        # -----------------------------------------
        # B. TEXT PROCESSING (Transformers)
        # -----------------------------------------
        # Tokenize using the DistilRoBERTa dictionary
        text_encoding = tokenizer(
            transcript,
            padding='max_length',
            truncation=True,
            max_length=MAX_TEXT_LENGTH,
            return_tensors='pt'
        )

        # -----------------------------------------
        # C. SAVE TO GOOGLE DRIVE
        # -----------------------------------------
        audio_save_path = os.path.join(FEATURES_DIR, f"{base_name}_audio.pt")
        text_save_path = os.path.join(FEATURES_DIR, f"{base_name}_text.pt")

        torch.save(audio_tensor, audio_save_path)
        torch.save(text_encoding, text_save_path)

    except Exception as e:
        # If one file breaks, don't crash the whole program! Just skip it.
        print(f"\n❌ Warning: Skipped file {base_name} due to error: {e}")

print(f"\n✅ Success! All features saved to {FEATURES_DIR}")
print("Data preprocessing is 100% complete. You are ready to train!")

--- Starting Multimodal Feature Extraction ---
Note: Saving thousands of files to Google Drive can take 15-30 minutes.
Please leave this tab open until it finishes!


Processing Files:   0%|          | 0/5600 [00:00<?, ?it/s]


✅ Success! All features saved to /content/drive/MyDrive/Emotion_Recognition_Project/Features
Data preprocessing is 100% complete. You are ready to train!


In [ ]:
!python models/speech_pipeline/train.py

--- Starting the AUDIO-ONLY Training Pipeline ---
Epoch 1/10 | Train Loss: 1.9248 | Validation Accuracy: 32.77%
Epoch 2/10 | Train Loss: 1.6861 | Validation Accuracy: 60.09%
Epoch 3/10 | Train Loss: 0.9354 | Validation Accuracy: 95.62%
Epoch 4/10 | Train Loss: 0.4559 | Validation Accuracy: 98.66%
Epoch 5/10 | Train Loss: 0.2313 | Validation Accuracy: 99.11%
Epoch 6/10 | Train Loss: 0.1335 | Validation Accuracy: 99.20%
Epoch 7/10 | Train Loss: 0.0830 | Validation Accuracy: 99.55%
Epoch 8/10 | Train Loss: 0.0571 | Validation Accuracy: 99.55%
Epoch 9/10 | Train Loss: 0.0421 | Validation Accuracy: 98.39%
Epoch 10/10 | Train Loss: 0.0396 | Validation Accuracy: 99.64%
Training Complete! Saved best_audio_model.pth


In [ ]:
!python models/speech_pipeline/test.py

In [ ]:
!python text_augmentation.py

config.json: 100% 1.00k/1.00k [00:00<00:00, 3.71MB/s]
tokenizer_config.json: 100% 294/294 [00:00<00:00, 2.00MB/s]
vocab.json: 100% 798k/798k [00:00<00:00, 26.5MB/s]
merges.txt: 100% 456k/456k [00:00<00:00, 105MB/s]
tokenizer.json: 100% 1.36M/1.36M [00:00<00:00, 113MB/s]
special_tokens_map.json: 100% 239/239 [00:00<00:00, 1.76MB/s]
Generating augmented text features...
Processing: 100% 5600/5600 [01:11<00:00, 77.79it/s]
✓ Generated augmented features for 5600 files
✓ Saved to: /content/drive/MyDrive/Emotion_Recognition_Project/TESS_Dataset/Features_Augmented
✓ Log saved to: /content/drive/MyDrive/Emotion_Recognition_Project/TESS_Dataset/Features_Augmented/augmentation_log.csv


In [ ]:
!python models/text_pipeline/train_augmented.py

TEXT-ONLY PIPELINE WITH AUGMENTED FEATURES
Device: cuda

Paths:
  CSV: /content/drive/MyDrive/Emotion_Recognition_Project/tess_metadata.csv
  Features: /content/drive/MyDrive/Emotion_Recognition_Project/TESS_Dataset/Features
  Augmented: /content/drive/MyDrive/Emotion_Recognition_Project/TESS_Dataset/Features_Augmented
  Checkpoints: /content/drive/MyDrive/Emotion_Recognition_Project/Checkpoints

[1/4] Loading dataset with augmented text features...
  Train samples: 4480
  Test samples: 1120
  Batches per epoch: 280

[2/4] Initializing TextClassifier...
Loading weights: 100% 103/103 [00:00<00:00, 2497.36it/s, Materializing param=pooler.dense.weight]
RobertaModel LOAD REPORT from: distilroberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight  

In [ ]:
!python models/text_pipeline/test_augmented.py

TESTING TEXT MODEL (Trained with Augmented Features)

Device: cuda

Paths:
  CSV: /content/drive/MyDrive/Emotion_Recognition_Project/tess_metadata.csv
  Features: /content/drive/MyDrive/Emotion_Recognition_Project/TESS_Dataset/Features
  Augmented: /content/drive/MyDrive/Emotion_Recognition_Project/TESS_Dataset/Features_Augmented
  Model: /content/drive/MyDrive/Emotion_Recognition_Project/best_text_model_augmented.pth

[1/3] Loading augmented test dataset...
  Test samples: 1120
  Test batches: 70

[2/3] Loading trained model...
Loading weights: 100% 103/103 [00:00<00:00, 4852.34it/s, Materializing param=pooler.dense.weight]
RobertaModel LOAD REPORT from: distilroberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

N

In [ ]:
!python models/fusion_pipeline/test_augmented.py

MULTIMODAL FUSION PIPELINE WITH AUGMENTED TEXT
Device: cuda

Paths:
  CSV: /content/drive/MyDrive/Emotion_Recognition_Project/tess_metadata.csv
  Features: /tmp/Features_Local
  Augmented: /tmp/Features_Augmented_Local
  Model save: /content/drive/MyDrive/Emotion_Recognition_Project/best_multimodal_model_augmented.pth

[1/4] Loading dataset with augmented text features...
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
  Train samples: 4480
  Validation samples: 1120
  Batches per epoch: 280

[2/4] Initializing MultimodalFusion model...
config.json: 100% 480/480 

In [ ]:
!python models/fusion_pipeline/test_augmented.py

TESTING MULTIMODAL FUSION (Trained with Augmented Text)

Device: cuda

Paths:
  CSV: /content/drive/MyDrive/Emotion_Recognition_Project/tess_metadata.csv
  Features: /tmp/Features_Local
  Augmented: /tmp/Features_Augmented_Local
  Model: /content/drive/MyDrive/Emotion_Recognition_Project/best_multimodal_model_augmented.pth

[1/3] Loading augmented multimodal dataset...
  Test samples: 1120
  Test batches: 70

[2/3] Loading trained model...
Loading weights: 100% 103/103 [00:00<00:00, 1199.73it/s, Materializing param=pooler.dense.weight]
RobertaModel LOAD REPORT from: distilroberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if